# HADA Dynamic Spatial per Scale — BCI IV-2a LOSO (S01-S09)
Enable **GPU** and **Internet** in Kaggle, then run the cell below.


In [ ]:
BRANCH = 'feature/hada-dynamic-spatial-scale'
!pip -q install uv
%cd /kaggle/working
!if [ -d tcformer-test/.git ]; then git -C tcformer-test remote set-url origin https://github.com/CuongDM1806/tcformer-test.git && git -C tcformer-test fetch origin $BRANCH && git -C tcformer-test checkout $BRANCH && git -C tcformer-test pull --ff-only origin $BRANCH; else git clone --branch $BRANCH --single-branch https://github.com/CuongDM1806/tcformer-test.git; fi
%cd /kaggle/working/tcformer-test
!git log -1 --oneline
!grep -nE 'ScaleConditionedDynamicSpatialConv|riemannian_alignment: true|im_tta_steps: 5' models/dynamic_spatial.py models/tcformer.py configs/hada_tcformer.yaml
!uv venv --clear --python 3.10 .venv
!uv pip install --python .venv/bin/python torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu126
!uv pip install --python .venv/bin/python -r requirements.txt
!nvidia-smi
!CUDA_VISIBLE_DEVICES=0 .venv/bin/python -c "import torch; from models.dynamic_spatial import ScaleConditionedDynamicSpatialConv as D; x=torch.randn(2,96,22,1000,device='cuda',requires_grad=True); m=D(96,3,22,2).cuda(); y=m(x); assert y.shape==(2,192,1,1000); y.square().mean().backward(); print('BCI2a dynamic-spatial CUDA smoke:',tuple(y.shape))"
!PYTHONUNBUFFERED=1 MPLBACKEND=Agg .venv/bin/python -u train_pipeline.py --model hada_tcformer --dataset bcic2a --loso --gpu_id 0
